# boolean-mask-identity-replace — faded example 3: Safe batched inverse via identity substitution

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `boolean-mask-identity-replace`. The last cell reports your progress on the `Numpy: Indexing and selection` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `boolean-mask-identity-replace`**, which bridges to the bank subtopic `Numpy: Indexing and selection` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-identity-replace"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Before a batched `linalg.inv`, replace degenerate slices so the whole batch does not raise. Detect them with `dets.abs() < eps` on `(B,)` determinants, identity-substitute the flagged `(N, N)` slots in a clone, then invert. Flagged slots come out as the identity (its own inverse).

## Faded exercise 3

### Safe batched inverse

Implement `safe_batched_inverse(A, eps=1e-6)` for a `(B, N, N)` float tensor `A`:

1. `dets = torch.linalg.det(A)` → shape `(B,)`.
2. Build `singular = dets.abs() < eps`.
3. Clone `A`, replace every flagged `(N, N)` submatrix with `torch.eye(N, dtype=A.dtype)`.
4. Return `torch.linalg.inv` of the cleaned tensor. Degenerate slots come out as the identity.

The det, mask, clone, and final inverse are written. Complete the **identity-substitution** step. `A` must NOT be mutated.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def safe_batched_inverse(A: Tensor, eps: float = 1e-6) -> Tensor:
    dets = t.linalg.det(A)
    singular = dets.abs() < eps
    N = A.shape[-1]
    A_safe = A.clone()
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
    return t.linalg.inv(A_safe)


def _test():
    t.manual_seed(0)
    A = t.randn(4, 3, 3)
    # force slot 1 to be exactly singular (a zero row)
    A[1, 0, :] = 0.0
    N = A.shape[-1]
    before = A.clone()
    out = safe_batched_inverse(A, eps=1e-6)
    dets = t.linalg.det(before)
    singular = dets.abs() < 1e-6
    assert singular[1].item(), 'fixture slot 1 should be singular'
    # flagged slot -> identity inverse -> identity
    assert t.allclose(out[singular], t.eye(N).expand(int(singular.sum()), N, N), atol=1e-5), 'flagged slot not identity'
    # non-flagged slots match a clean reference inverse
    ref = t.linalg.inv(before[~singular])
    assert t.allclose(out[~singular], ref, atol=1e-4), 'good slots differ from true inverse'
    assert t.equal(A, before), 'input A was mutated'
    assert out.shape == A.shape


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def safe_batched_inverse(A: Tensor, eps: float = 1e-6) -> Tensor:
    dets = t.linalg.det(A)
    singular = dets.abs() < eps
    N = A.shape[-1]
    A_safe = A.clone()
    A_safe[singular] = t.eye(N, dtype=A.dtype)
    return t.linalg.inv(A_safe)
```
</details>